# 02 — 8-week production simulation

Loops through 8 weeks of the Home Credit dataset. Week 0 bootstraps the
initial `@champion`. Weeks 1-7 each train a challenger, evaluate against
the champion, and the gate decides: promote, or keep the current champion?

**Drift injection** kicks in from week 4 onward: `AMT_INCOME_TOTAL` is
multiplied by 1.10 with Gaussian noise. We expect the system to either
adapt (promote a new champion that handles the new distribution) or
refuse (if fairness or drift gates trip).

In [ ]:
%load_ext autoreload
%autoreload 2

import shutil
from pathlib import Path
import pandas as pd

from modelgate import audit
from modelgate.config import settings
from modelgate.retrain import run_retrain

## Reset state so the sim is reproducible

In [ ]:
for p in [Path('mlruns.db'), Path('logs/decisions.jsonl')]:
    if p.exists():
        p.unlink()
for d in [Path('mlruns'), Path('mlartifacts')]:
    if d.exists():
        shutil.rmtree(d)
        d.mkdir()
print('state reset')

## Run the 8 weeks

In [ ]:
history = []
for week in range(settings.n_weeks):
    print(f'\n========== WEEK {week} ==========')
    record = run_retrain(week=week)
    history.append(record)
    print(f'  -> {record["action"]}')
    if 'decision' in record and record['decision']['reasons']:
        for r in record['decision']['reasons']:
            print(f'     skip reason: {r}')
print('\nsim complete')

## Decision timeline

In [ ]:
rows = []
for h in history:
    row = {
        'week': h['week'],
        'action': h['action'],
        'challenger_version': h.get('challenger_version'),
    }
    if 'challenger_metrics' in h:
        row['challenger_auc'] = h['challenger_metrics']['roc_auc']
        row['champion_auc'] = h['champion_metrics']['roc_auc']
        chal_dp = min(h['challenger_fairness']['dp_ratio_by_attr'].values()) if h['challenger_fairness']['dp_ratio_by_attr'] else None
        champ_dp = min(h['champion_fairness']['dp_ratio_by_attr'].values()) if h['champion_fairness']['dp_ratio_by_attr'] else None
        row['challenger_min_dp'] = chal_dp
        row['champion_min_dp'] = champ_dp
        row['train_drift_psi'] = h['train_drift_psi']
    rows.append(row)
timeline = pd.DataFrame(rows)
timeline

## Plot AUC over weeks

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax = axes[0]
tl = timeline.dropna(subset=['challenger_auc'])
ax.plot(tl['week'], tl['challenger_auc'], 'o-', label='challenger AUC')
ax.plot(tl['week'], tl['champion_auc'], 's--', label='champion AUC')
for i, row in tl.iterrows():
    if row['action'] == 'promote':
        ax.axvline(row['week'], color='green', alpha=0.2)
ax.axvline(settings.drift_injection_start_week, color='red', linestyle=':', label='drift injected')
ax.set_ylabel('AUC')
ax.set_title('Challenger vs champion AUC (green = promoted)')
ax.legend()

ax = axes[1]
ax.plot(tl['week'], tl['challenger_min_dp'], 'o-', label='challenger min DP ratio')
ax.plot(tl['week'], tl['champion_min_dp'], 's--', label='champion min DP ratio')
ax.axhline(settings.dp_ratio_min, color='red', linestyle='--', label=f'gate floor {settings.dp_ratio_min}')
ax.axvline(settings.drift_injection_start_week, color='red', linestyle=':')
ax.set_xlabel('week')
ax.set_ylabel('min DP ratio across protected attrs')
ax.set_title('Fairness regression guard')
ax.legend()
plt.tight_layout()
plt.show()

## Audit log

In [ ]:
for entry in audit.read_all():
    if entry['action'] == 'bootstrap_champion':
        print(f"week {entry['week']}: BOOTSTRAP v{entry['version']}")
    elif entry['action'] == 'promote':
        print(f"week {entry['week']}: PROMOTE v{entry['challenger_version']} (was v{entry['champion_version_before']})")
    elif entry['action'] == 'skip':
        reasons = '; '.join(c['name'] for c in entry['decision']['checks'] if not c['passed'])
        print(f"week {entry['week']}: skip v{entry['challenger_version']} -- failed: {reasons}")